# Reducto-style onsite mock: Receipt extraction
公开模型与 CORD-v2 的模拟项目，不是 Reducto 真题。先启用 GPU runtime。建议计时前完成安装和下载。

上传项目 zip 后先阅读 BRIEF.md。使用 TRL SFTTrainer + Qwen2.5-VL-3B 4-bit LoRA；默认只使用一张 GPU。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, os
uploaded = files.upload()  # Choose reducto-onsite-receipt-lab.zip
for name, data in uploaded.items():
    if not name.endswith('.zip'):
        continue
    destination = Path('/content').resolve()
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination) or ((member.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Unsafe zip entry')
            if target.exists() and not member.is_dir():
                raise FileExistsError(f'Refusing to overwrite {target}')
        z.extractall(destination)
os.chdir('/content/onsite-receipt-lab')
print(Path('BRIEF.md').read_text())


## 安装固定版本
这会调整 Colab 的 torch/torchvision/torchaudio 和相关依赖。安装完成后使用菜单 **Runtime → Restart session**，再从下一个单元格继续，不要重复上传覆盖项目。驱动须支持 CUDA 12.4。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', '/content/onsite-receipt-lab/requirements-gpu.txt'])
print('Restart session now, then continue below.')


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.chdir('/content/onsite-receipt-lab')
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first'
print('torch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory/2**30)
print('BF16 supported:', torch.cuda.is_bf16_supported())
def run(*args):
    subprocess.check_call([sys.executable, *args])
run('-m', 'pytest', '-q', 'tests')


## 下载真实数据与权重
基础权重约 7.51 GB，另需数据、缓存、adapter 空间。prepare 拒绝覆盖已有 manifests；已有完整 data/cord 时可跳过该条命令。检查 provenance 和 excluded.jsonl，了解单值 schema 排除了哪些多值标注。

In [ ]:
run('-m', 'receipt_lab.prepare', '--output', 'data/cord')
run('-m', 'receipt_lab.prefetch')
print(Path('data/cord/provenance.json').read_text())


## 人工核对一条训练样本
检查输出字段是否与图片一致，尤其是行价格、数量和金额分隔符。

In [ ]:
from IPython.display import display
from PIL import Image
row = json.loads(Path('data/cord/train.jsonl').read_text().splitlines()[0])
display(Image.open(Path('data/cord') / row['image_path']))
print(json.dumps(row['target'], ensure_ascii=False, indent=2))


## Baseline：先两张 smoke，再完整 validation
记录实际单张耗时和显存。不要把两张样本的指标当正式结果。

In [ ]:
run('-m','receipt_lab.predict','--limit','2','--output','runs/baseline-smoke.jsonl')
run('-m','receipt_lab.evaluate','--limit','2','--predictions','runs/baseline-smoke.jsonl','--output','runs/baseline-smoke.metrics.json')


In [ ]:
run('-m','receipt_lab.predict','--output','runs/baseline.jsonl')
run('-m','receipt_lab.evaluate','--predictions','runs/baseline.jsonl','--output','runs/baseline.metrics.json')


## 一次真实 GPU smoke 更新和 adapter 重载
先确认资源和实现可用，再运行正式实验。T4 自动使用 FP16；支持 BF16 的卡使用 BF16。这里的两步用于验证，不代表完成模型改进。

In [ ]:
run('-m','receipt_lab.train','--limit','8','--max-steps','2','--output','runs/gpu-smoke')
run('-m','receipt_lab.predict','--adapter','runs/gpu-smoke','--limit','2','--output','runs/gpu-smoke-reloaded.jsonl')


## 正式训练实验
先写假设和 baseline 错误分析，再决定是否修改 configs。默认 100 optimizer steps；运行时间必须实测。每个实验使用新的输出目录。

In [ ]:
run('-m','receipt_lab.train','--output','runs/lora-v1')
run('-m','receipt_lab.predict','--adapter','runs/lora-v1','--output','runs/lora-v1.validation.jsonl')
run('-m','receipt_lab.evaluate','--predictions','runs/lora-v1.validation.jsonl','--output','runs/lora-v1.metrics.json')


## 检查真实预测并对比
静态 review HTML 使用相对原图路径。Notebook 可以直接展示原图与两个 raw outputs；不要仅展示成功样本。

In [ ]:
from receipt_lab.common import load_rows
from receipt_lab.evaluate import read_predictions, score_one
rows = load_rows('data/cord/validation.jsonl')
base = read_predictions('runs/baseline.jsonl')
candidate = read_predictions('runs/lora-v1.validation.jsonl')
for row in rows[:3]:
    display(Image.open(row['image_path']))
    print('ID', row['id'], '\nTARGET', row['target'])
    for label, predictions in [('baseline', base), ('candidate', candidate)]:
        print(label, score_one(row['target'], predictions.get(row['id'], {})))
        print(predictions.get(row['id'], {}).get('raw_text', 'MISSING'))
run('-m','receipt_lab.report','--baseline','runs/baseline.jsonl','--candidate','runs/lora-v1.validation.jsonl','--output','runs/comparison.html')


## 剩余工作由你完成
1. 打开 INTERVIEW_EVENTS.md 对应的追加需求。
2. 做至少一个可解释的改进/消融，重新评估。
3. 按 README 在新进程验证本地推理接口。
4. 冻结方案后才跑 test。
5. 填写 EXPERIMENT_LOG.md，准备 15 分钟 review。

不要将数据和模型权重提交 Git。Colab session 可能丢失，及时保存 adapter、日志和配置。下面导出 runs 和实验记录，不包含基础模型或完整原图。

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('/content/onsite-results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for path in Path('runs').rglob('*'):
        if path.is_file(): z.write(path, str(path))
    z.write('EXPERIMENT_LOG.md')
files.download('/content/onsite-results.zip')
